### New crawler

In [6]:
import requests
import pandas as pd
import time
import random
import re
import json
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed

class CourseraFullScraper:
    def __init__(self):
        self.base_api_url = "https://api.coursera.org/api/courses.v1"
        self.filename = "coursera_full_database.csv"
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
            "Accept-Language": "en-US,en;q=0.9"
        }
        self.session = requests.Session()
        self.session.headers.update(self.headers)

    def fetch_course_index(self):
        """Step 1: Fetch ALL courses by iterating through the API pagination."""
        print("🚀 [Step 1/2] Fetching complete course index from API...")
        all_courses = []
        start = 0
        batch_size = 100 
        
        fields = "name,slug,workload,primaryLanguages,difficultyLevel,difficulty,domainTypes,partnerIds,courseType"
        includes = "partnerIds"

        while True: # Removed the 'limit' constraint
            params = {
                "start": start,
                "limit": batch_size,
                "fields": fields,
                "includes": includes
            }
            
            try:
                response = self.session.get(self.base_api_url, params=params, timeout=15)
                if response.status_code != 200: 
                    print(f"\n⚠️ API stopped responding at index {start} (Status: {response.status_code})")
                    break
                
                data = response.json()
                elements = data.get('elements', [])
                
                # The exit condition: if the 'elements' list is empty, we reached the end
                if not elements: 
                    break

                partners_map = {p['id']: p['name'] for p in data.get('linked', {}).get('partners.v1', [])}

                for item in elements:
                    diff = item.get("difficultyLevel") or item.get("difficulty", "N/A")
                    
                    course_data = {
                        "Course title": item.get("name"),
                        "Course_Type": item.get("courseType", "Course"),
                        "Organization": partners_map.get(item.get('partnerIds', [None])[0], "Coursera"),
                        "Workload": item.get("workload", "N/A"),
                        "Difficulty_level": diff,
                        "Language": ",".join(item.get("primaryLanguages", [])),
                        "Course_link": f"https://www.coursera.org/learn/{item.get('slug')}",
                        "Ratings": "N/A",
                        "Review count": 0
                    }
                    all_courses.append(course_data)
                
                start += batch_size
                print(f"   Indexed {len(all_courses)} courses...", end='\r')
                time.sleep(0.4) # Slight delay to be polite
                
            except Exception as e: 
                print(f"\n❌ Error during API sync: {e}")
                break
                
        print(f"\n✅ Indexing complete. Total courses found: {len(all_courses)}")
        return all_courses

    def scrape_detailed_stats(self, course_data):
        """Step 2: Scrape details for each course."""
        url = course_data["Course_link"]
        try:
            time.sleep(random.uniform(0.5, 1.2)) # Random delay to prevent IP block
            response = self.session.get(url, timeout=15)
            if response.status_code == 200:
                content = response.text
                
                # Regex for Ratings/Reviews
                rating_match = re.search(r'"ratingValue"\s*:\s*([\d\.]+)', content)
                review_match = re.search(r'"reviewCount"\s*:\s*(\d+)', content)
                if rating_match: course_data["Ratings"] = round(float(rating_match.group(1)), 1)
                if review_match: course_data["Review count"] = int(review_match.group(1))

                # Regex for Difficulty fallback
                if course_data["Difficulty_level"] == "N/A":
                    diff_match = re.search(r'"difficultyLevel"\s*:\s*"([^"]+)"', content)
                    if diff_match:
                        course_data["Difficulty_level"] = diff_match.group(1).capitalize()
                    else:
                        for level in ["Beginner", "Intermediate", "Advanced", "Mixed"]:
                            if re.search(rf'{level}\s*Level', content, re.IGNORECASE):
                                course_data["Difficulty_level"] = level
                                break
        except Exception: pass
        return course_data

    def run(self):
        # 1. Get the list
        courses = self.fetch_course_index()
        
        # 2. Enrich the data
        print("\n🚀 [Step 2/2] Scrapping detailed data (This will take a while)...")
        enriched_data = []
        
        # Max workers kept at 5 to avoid triggering anti-bot protection
        with ThreadPoolExecutor(max_workers=5) as executor:
            future_to_course = {executor.submit(self.scrape_detailed_stats, c): c for c in courses}
            processed = 0
            total = len(courses)
            for future in as_completed(future_to_course):
                enriched_data.append(future.result())
                processed += 1
                if processed % 20 == 0:
                    print(f"   Progress: {processed}/{total} processed...", end='\r')

        df = pd.DataFrame(enriched_data)        
        df_for_stats = df.replace("N/A", pd.NA)
        self.print_summary(df_for_stats)
        df.to_csv(self.filename, index=False, encoding='utf_8_sig')
        print(f"\n\n✅ Done! Data saved to {self.filename}")

    def print_summary(self, df):
        """Displays missing values and value distributions."""
        print("\n" + "="*60)
        print("📊 DATASET ANALYSIS")
        print("="*60)
        
        # Missing values and unique counts
        summary = pd.DataFrame({
            'Missing (Null)': df.isnull().sum(),
            'Unique Values': df.nunique()
        })
        print(summary)
        
        # Value distribution for key columns
        cols_to_check = ["Course_Type", "Difficulty_level", "Language"]
        for col in cols_to_check:
            print(f"\n--- Distribution for: {col} ---")
            print(df[col].value_counts().head(10)) # Shows top 10 categories
        
        print("="*60)

if __name__ == "__main__":
    scraper = CourseraFullScraper()
    scraper.run()
    

🚀 [Step 1/2] Fetching complete course index from API...
   Indexed 18902 courses...
✅ Indexing complete. Total courses found: 18902

🚀 [Step 2/2] Scrapping detailed data (This will take a while)...
   Progress: 18900/18902 processed...
📊 DATASET ANALYSIS
                  Missing (Null)  Unique Values
Course title                   0          18793
Course_Type                    0              2
Organization                   0            374
Workload                       0           3664
Difficulty_level            2161              3
Language                       0             26
Course_link                    0          18902
Ratings                     9850             21
Review count                   0           1812

--- Distribution for: Course_Type ---
Course_Type
v2.ondemand    18782
v2.capstone      120
Name: count, dtype: int64

--- Distribution for: Difficulty_level ---
Difficulty_level
Beginner        9823
Intermediate    6198
Advanced         720
Name: count, dtype: in

In [7]:
df = pd.read_csv("coursera_full_database.csv")
print(df['Course_Type'].value_counts())

Course_Type
v2.ondemand    18782
v2.capstone      120
Name: count, dtype: int64
